
This notebook collects daily OHLCV data, cleans price and volume data, constructs a tradable universe, creates basic alpha features, builds future return labels, and saves the processed dataset for model training.

Limitation: This project uses the current S&P 1500 constituents
for the full historical sample, so backtest results may contain
survivorship bias.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import yfinance as yf
import requests
from io import StringIO

PROJECT_NAME = "alpha_ml"
BASE_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME

RAW_DATA_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"
FEATURES_DIR = BASE_DIR / "data" / "features"

CONFIG_PATH = BASE_DIR / "config.json"

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

config

Mounted at /content/drive


{'start_date': '2015-01-01',
 'end_date': '2025-12-31',
 'prediction_horizon': 5,
 'min_price': 5.0,
 'liquidity_window': 20,
 'min_history_days': 252,
 'return_windows': [1, 3, 5, 10, 20, 60],
 'volatility_windows': [20, 60],
 'moving_average_windows': [20, 60],
 'rebalance_frequency': 5,
 'top_k': 50,
 'transaction_cost_bps': 10,
 'test_years': [2021, 2022, 2023, 2024]}

In [2]:
# create sp1500
headers = {
    "User-Agent": "Mozilla/5.0"
}

def read_wiki_table(url, symbol_col="Symbol"):
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    tables = pd.read_html(StringIO(response.text))

    for table in tables:
        if symbol_col in table.columns:
            return table, table[symbol_col].dropna().astype(str).tolist()

    raise ValueError(f"Could not find table with column: {symbol_col}")

sp500_url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
sp400_url = "https://en.wikipedia.org/wiki/List_of_S%26P_400_companies"
sp600_url = "https://en.wikipedia.org/wiki/List_of_S%26P_600_companies"

sp500_table, sp500_tickers = read_wiki_table(sp500_url, symbol_col="Symbol")
sp400_table, sp400_tickers = read_wiki_table(sp400_url, symbol_col="Symbol")
sp600_table, sp600_tickers = read_wiki_table(sp600_url, symbol_col="Symbol")

tickers = sp500_tickers + sp400_tickers + sp600_tickers

tickers = sorted(set(tickers))
tickers = [ticker.replace(".", "-") for ticker in tickers]

sp500_table["source_index"] = "S&P 500"
sp400_table["source_index"] = "S&P MidCap 400"
sp600_table["source_index"] = "S&P SmallCap 600"

universe_table = pd.concat(
    [sp500_table, sp400_table, sp600_table],
    ignore_index=True
)

universe_table["Symbol"] = (
    universe_table["Symbol"]
    .astype(str)
    .str.replace(".", "-", regex=False)
)

universe_table = universe_table.drop_duplicates(subset=["Symbol"])

print(len(tickers), tickers[:10])

# save sp1500 universe
universe_path = RAW_DATA_DIR / "sp1500_universe.csv"
universe_table.to_csv(universe_path, index=False)
print(f"Universe saved to: {universe_path}")

1506 ['A', 'AA', 'AAL', 'AAMI', 'AAON', 'AAP', 'AAPL', 'AAT', 'ABBV', 'ABCB']
Universe saved to: /content/drive/MyDrive/alpha_ml/data/raw/sp1500_universe.csv


In [3]:
# download data from yf
start_date = config["start_date"]
end_date = config["end_date"]

raw_data = yf.download(
    tickers=tickers,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    group_by="ticker",
    threads=True,
    progress=True
)

print(raw_data.head())

# save data
raw_price_path = RAW_DATA_DIR / "sp1500_ohlcv_raw.parquet"
raw_data.to_parquet(raw_price_path)
print(f"Raw OHLCV data saved to: {raw_price_path}")

[**********************94%********************   ]  1421 of 1506 completedERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CWEN-A"}}}
[*********************100%***********************]  1506 of 1506 completed
ERROR:yfinance:
8 Failed downloads:
ERROR:yfinance:['MFP', 'VGNT', 'HONA', 'LEG', 'ADIG', 'FDXF', 'MBGL']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-01-01 -> 2025-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 1420088400, endDate = 1767157200")')
ERROR:yfinance:['CWEN-A']: YFTzMissingError('possibly delisted; no timezone found')


Ticker           SYNA                                                       \
Price            Open       High        Low      Close  Adj Close   Volume   
Date                                                                         
2015-01-02  66.269997  66.980003  62.750000  64.250000  64.250000  1612300   
2015-01-05  64.099998  65.769997  63.160000  63.720001  63.720001   889900   
2015-01-06  64.570000  65.000000  60.060001  60.430000  60.430000  1124500   
2015-01-07  60.660000  61.720001  60.660000  61.570000  61.570000   610700   
2015-01-08  60.610001  62.599998  60.430000  60.840000  60.840000  1237500   

Ticker           WERN                                   ...        TJX  \
Price            Open       High        Low      Close  ...        Low   
Date                                                    ...              
2015-01-02  31.340000  31.549999  30.570000  31.129999  ...  33.755001   
2015-01-05  31.090000  31.090000  30.440001  30.490000  ...  33.514999   
2015-

In [4]:
# Change format
records = []

for ticker in tickers:
    if ticker not in raw_data.columns.get_level_values(0):
        continue

    df = raw_data[ticker].copy()
    df["ticker"] = ticker
    df["date"] = df.index

    df = df.rename(columns={
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Adj Close": "adj_close",
        "Volume": "volume"
    })

    records.append(df)

model_df_full = pd.concat(records, axis=0, ignore_index=True)

model_df_full = model_df_full[
    ["date", "ticker", "open", "high", "low", "close", "adj_close", "volume"]
]

model_df_full.head()

Price,date,ticker,open,high,low,close,adj_close,volume
0,2015-01-02,A,41.180000,41.310001,40.369999,40.560001,36.899399,1529200.0
1,2015-01-05,A,40.320000,40.459999,39.700001,39.799999,36.207977,2041800.0
2,2015-01-06,A,39.810001,40.020000,39.020000,39.180000,35.643921,2080600.0
3,2015-01-07,A,39.520000,39.810001,39.290001,39.700001,36.117008,3359700.0
4,2015-01-08,A,40.240002,40.980000,40.180000,40.889999,37.199608,2116300.0


In [5]:
# basic cleaning
model_df_full = model_df_full.sort_values(["ticker", "date"]).reset_index(drop=True)

model_df_full = model_df_full[model_df_full["adj_close"] > 0]
model_df_full = model_df_full[model_df_full["volume"] >= 0]

print("Date range:", model_df_full["date"].min(), "to", model_df_full["date"].max())
print("Number of tickers:", model_df_full["ticker"].nunique())
print("Number of rows:", len(model_df_full))

model_df_full.isna().mean()

Date range: 2015-01-02 00:00:00 to 2025-12-30 00:00:00
Number of tickers: 1498
Number of rows: 3828683


,0
Price,
date,0.0
ticker,0.0
open,0.0
high,0.0
low,0.0
close,0.0
adj_close,0.0
volume,0.0


In [6]:
# liquidity
model_df_full["dollar_volume"] = model_df_full["close"] * model_df_full["volume"]

model_df_full["dollar_volume_20d"] = (
    model_df_full.groupby("ticker")["dollar_volume"]
    .rolling(config["liquidity_window"])
    .mean()
    .reset_index(level=0, drop=True)
)

model_df_full["history_days"] = model_df_full.groupby("ticker").cumcount() + 1

# construct universe
min_price = config["min_price"]
min_history_days = config["min_history_days"]

model_df_full["is_tradable"] = (
    (model_df_full.groupby("ticker")["close"].shift(1) >= min_price) &
    (model_df_full["history_days"] >= min_history_days) &
    (model_df_full.groupby("ticker")["dollar_volume_20d"].shift(1).notna())
)
print("tradable ratio: ", model_df_full["is_tradable"].mean())

model_df_full["liquidity_rank"] = (
    model_df_full.groupby("ticker")["dollar_volume_20d"].shift(1)
    .where(model_df_full["is_tradable"])
    .groupby(model_df_full["date"])
    .rank(ascending=False, method="first")
)

model_df_full["in_universe"] = (
    model_df_full["is_tradable"] &
    (model_df_full["liquidity_rank"] <= 500)
)

print("universe", model_df_full["in_universe"].mean())

tradable ratio:  0.8936260850010304
universe 0.3283113279422715


In [7]:
# features use current data; shift by 1 for model input.
for window in config["return_windows"]:
    model_df_full[f"ret_{window}d"] = (
        model_df_full.groupby("ticker")["adj_close"]
        .pct_change(window)
    )

for window in config["volatility_windows"]:
    model_df_full[f"vol_{window}d"] = (
        model_df_full.groupby("ticker")["ret_1d"]
        .rolling(window)
        .std()
        .reset_index(level=0, drop=True)
    )

for window in config["moving_average_windows"]:
    ma = (
        model_df_full.groupby("ticker")["adj_close"]
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

    model_df_full[f"close_to_ma_{window}d"] = model_df_full["adj_close"] / ma - 1

model_df_full["volume_20d_mean"] = (
    model_df_full.groupby("ticker")["volume"]
    .rolling(20)
    .mean()
    .reset_index(level=0, drop=True)
)

model_df_full["volume_relative_20d"] = (
    model_df_full["volume"] / model_df_full["volume_20d_mean"] - 1
)

# cross-sectional feature ranking
feature_cols = ["open", "high", "low", "close", "adj_close", "volume", "dollar_volume", "dollar_volume_20d"
] + [f"ret_{w}d" for w in config["return_windows"]
] + [f"vol_{w}d" for w in config["volatility_windows"]
] + [f"close_to_ma_{w}d" for w in config["moving_average_windows"]
] + ["volume_20d_mean", "volume_relative_20d"]

for col in feature_cols:
    model_df_full[f"{col}_rank"] = (
        model_df_full.groupby("date")[col]
        .rank(pct=True)
    )

rank_feature_cols = [f"{col}_rank" for col in feature_cols]
feature_cols += ["liquidity_rank"]

In [8]:
# construct target labels
horizon = config["prediction_horizon"]

model_df_full[f"future_{horizon}d_return"] = (
    model_df_full.groupby("ticker")["adj_close"]
    .shift(-horizon) / model_df_full["adj_close"] - 1
)

model_df_full[f"market_future_{horizon}d_return"] = (
    model_df_full[f"future_{horizon}d_return"]
    .where(model_df_full["in_universe"])
    .groupby(model_df_full["date"])
    .transform("mean")
)

model_df_full[f"future_{horizon}d_relative_return"] = (
    model_df_full[f"future_{horizon}d_return"] -
    model_df_full[f"market_future_{horizon}d_return"]
)
target_col = f"future_{horizon}d_relative_return"

In [9]:
# final dataset
model_df = model_df_full[
    model_df_full["in_universe"]
].copy()

model_df = model_df.dropna(subset=feature_cols + rank_feature_cols + [target_col])

model_df = model_df[
    ["date", "ticker"]
    + feature_cols
    + rank_feature_cols
    + [target_col]
]

print(model_df_full.head())
print(model_df.head())

print("Date range:", model_df["date"].min(), "to", model_df["date"].max())
print("Tickers:", model_df["ticker"].nunique())
print("Rows:", len(model_df))

model_df.describe()

Price       date ticker       open       high        low      close  \
0     2015-01-02      A  41.180000  41.310001  40.369999  40.560001   
1     2015-01-05      A  40.320000  40.459999  39.700001  39.799999   
2     2015-01-06      A  39.810001  40.020000  39.020000  39.180000   
3     2015-01-07      A  39.520000  39.810001  39.290001  39.700001   
4     2015-01-08      A  40.240002  40.980000  40.180000  40.889999   

Price  adj_close     volume  dollar_volume  dollar_volume_20d  ...  \
0      36.899399  1529200.0   6.202435e+07                NaN  ...   
1      36.207977  2041800.0   8.126364e+07                NaN  ...   
2      35.643921  2080600.0   8.151791e+07                NaN  ...   
3      36.117008  3359700.0   1.333801e+08                NaN  ...   
4      37.199608  2116300.0   8.653551e+07                NaN  ...   

Price  ret_60d_rank  vol_20d_rank  vol_60d_rank  close_to_ma_20d_rank  \
0               NaN           NaN           NaN                   NaN   
1     

Price,date,open,high,low,close,adj_close,volume,dollar_volume,dollar_volume_20d,ret_1d,...,ret_10d_rank,ret_20d_rank,ret_60d_rank,vol_20d_rank,vol_60d_rank,close_to_ma_20d_rank,close_to_ma_60d_rank,volume_20d_mean_rank,volume_relative_20d_rank,future_5d_relative_return
count,1254500,1.254500e+06,1.254500e+06,1.254500e+06,1.254500e+06,1.254500e+06,1.254500e+06,1.254500e+06,1.254500e+06,1.254500e+06,...,1.254500e+06,1.254500e+06,1.254500e+06,1.254500e+06,1.254500e+06,1.254500e+06,1.254500e+06,1.254500e+06,1.254500e+06,1.254500e+06
mean,2020-12-25 07:23:39.051415552,1.322557e+02,1.338568e+02,1.306165e+02,1.322592e+02,1.235790e+02,6.591473e+06,4.659970e+08,4.644465e+08,6.080643e-04,...,5.014109e-01,5.026283e-01,5.067455e-01,4.455045e-01,4.328135e-01,5.019491e-01,5.053179e-01,7.447792e-01,5.046653e-01,4.035566e-20
min,2015-12-31 00:00:00,3.800000e+00,4.266500e+00,3.550000e+00,3.800000e+00,2.389856e+00,0.000000e+00,0.000000e+00,2.363005e+07,-6.000000e-01,...,6.680027e-04,6.680027e-04,6.693440e-04,6.693440e-04,6.693440e-04,6.680027e-04,6.688963e-04,6.688963e-04,6.680027e-04,-8.556473e-01
25%,2018-06-28 00:00:00,4.534000e+01,4.592000e+01,4.474750e+01,4.534000e+01,3.917809e+01,1.224800e+06,1.148103e+08,1.296000e+08,-9.063436e-03,...,2.565407e-01,2.556861e-01,2.620056e-01,1.819398e-01,1.722950e-01,2.561124e-01,2.583140e-01,6.248227e-01,2.750000e-01,-2.007587e-02
50%,2020-12-23 00:00:00,8.277000e+01,8.375000e+01,8.177000e+01,8.277000e+01,7.379932e+01,2.492400e+06,1.985606e+08,2.091790e+08,7.829267e-04,...,5.036987e-01,5.054348e-01,5.117687e-01,4.160713e-01,3.959866e-01,5.044185e-01,5.099383e-01,7.843427e-01,5.056861e-01,-3.504248e-04
75%,2023-06-23 00:00:00,1.528700e+02,1.546600e+02,1.510400e+02,1.528800e+02,1.409056e+02,5.599125e+06,3.905121e+08,3.921652e+08,1.043796e-02,...,7.467705e-01,7.501859e-01,7.516981e-01,6.956822e-01,6.781273e-01,7.483589e-01,7.525703e-01,9.028910e-01,7.351145e-01,1.942785e-02
max,2025-12-22 00:00:00,9.914170e+03,9.964770e+03,9.794000e+03,9.924400e+03,9.924400e+03,1.963520e+09,1.543777e+11,5.443374e+10,1.348358e+00,...,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,7.904365e+00
std,NaN,2.449520e+02,2.478686e+02,2.421169e+02,2.449888e+02,2.439452e+02,2.225159e+07,1.572170e+09,1.465540e+09,2.348856e-02,...,2.845019e-01,2.857142e-01,2.838792e-01,2.944604e-01,2.931039e-01,2.848706e-01,2.853165e-01,1.937752e-01,2.744085e-01,4.464893e-02


In [10]:
# save processed dataset
model_df.to_parquet(FEATURES_DIR / "model_dataset.parquet", index=False)
model_df_full.to_parquet(FEATURES_DIR / "model_dataset_full.parquet", index=False)

with open(FEATURES_DIR / "feature_columns.json", "w") as f:
    json.dump(feature_cols + rank_feature_cols, f, indent=4)